# Notebook 1: QSARmil multi-conformer modelling

This notebook introduces the high-level API for building multi-conformer models with **QSARmil**. The API is designed to simplify benchmarking QSARmil against alternative approaches without requiring detailed knowledge of the underlying modelling configuration. As a result, the workflow remains concise and easy to follow. You can adapt the code below to your own dataset and run the complete **QSARmil** modelling pipeline with minimal modifications.

In [1]:
import pandas as pd

from sklearn.metrics import r2_score

### 1. Data load

As an example, we use a publicly available and easily accessible collection of molecular bioactivity datasets introduced in the paper by:

> Van Tilborg, Derek, Alisa Alenicheva, and Francesca Grisoni. "Exposing the limitations of molecular machine learning with activity cliffs." Journal of chemical information and modeling 62.23 (2022): 5938-5951.

From this collection, we select one dataset for demonstration purposes.

In [2]:
# load data
url = "https://raw.githubusercontent.com/molML/MoleculeACE/main/MoleculeACE/Data/benchmark_data/CHEMBL2034_Ki.csv"
df_ace = pd.read_csv(url)

# train/test split
df_train = df_ace[df_ace["split"] == "train"][["smiles", "y"]]
df_test = df_ace[df_ace["split"] == "test"][["smiles", "y"]]

In [3]:
# uncomment for quick testing
df_train = df_train.sample(frac=0.1, random_state=42).reset_index(drop=True)
df_test = df_test.sample(frac=0.1, random_state=42).reset_index(drop=True)
df_train.shape, df_test.shape

((60, 2), (15, 2))

### 2. Data validation

The data validation step is straightforward but includes an additional requirement compared to standard 2D modelling pipelines. In most QSAR workflows, input structures are primarily validated by checking whether their SMILES strings are valid and can be successfully parsed into molecular objects.

In contrast, **QSARmil** introduces an extra validation step: verifying whether a valid 3D structure can be generated for each molecule. Molecules that fail this step cannot be used in the 3D modelling pipeline. As a result, the filtering (removal) rate in QSARmil may be higher than in typical 2D QSAR workflows.

In [4]:
from qsarmil.data.input_data import DataValidator

In [5]:
dvalid = DataValidator(num_cpu=30, verbose=True)

df_train = dvalid.filter_dataframe(df_train)
df_test = dvalid.filter_dataframe(df_test)

No rows removed. All molecules are valid.
No rows removed. All molecules are valid.


### 3. Building model

The multi-conformer model building pipeline consists of several sequential steps:

 - Conformer generation: the maximum number of conformers is defined using the ``num_conf`` parameter.
 - Descriptor calculation: by default, multiple types of 3D molecular descriptors are computed.
 - Model training: several multi-instance learning networks are used to train models. Internal stepwise hyperparameter optimization can be enabled with ``hopt=True`` (note that this increases runtime).
 - Consensus search: once multiple multi-conformer models are trained, a genetic algorithm is applied to identify an optimal consensus model.

Task type is defined automatically (regression or binary classification), ``output_folder`` (directory for storing predictions from individual models), and ``verbose`` (controls the level of pipeline progress output).

In [6]:
from qsarmil.meta import MultiConformerModel

In [7]:
model = MultiConformerModel(num_conf=10, hopt=False, verbose=True, output_folder="./mcfm")
df_pred = model.run_predict(df_train, df_test)

No rows removed. All molecules are valid.
No rows removed. All molecules are valid.
No rows removed. All molecules are valid.
Generating conformers: 48/48
Generating conformers: 12/12
Generating conformers: 12/12
[1/72] Running model: RDKitGEOM|MeanInstanceWrapperMLPNetworkRegressor
  > Finished in 0.02 min | Memory usage: 1.156 GB
[2/72] Running model: RDKitGEOM|MeanBagWrapperMLPNetworkRegressor
  > Finished in 0.01 min | Memory usage: 1.157 GB
[3/72] Running model: RDKitGEOM|MeanBagNetworkRegressor
  > Finished in 0.00 min | Memory usage: 1.160 GB
[4/72] Running model: RDKitGEOM|MeanInstanceNetworkRegressor
  > Finished in 0.00 min | Memory usage: 1.162 GB
[5/72] Running model: RDKitGEOM|AdditiveAttentionNetworkRegressor
  > Finished in 0.01 min | Memory usage: 1.163 GB
[6/72] Running model: RDKitGEOM|SelfAttentionNetworkRegressor
  > Finished in 0.01 min | Memory usage: 1.165 GB
[7/72] Running model: RDKitGEOM|HopfieldAttentionNetworkRegressor
  > Finished in 0.01 min | Memory usage

In [8]:
r2_score(df_test["y"], df_pred["pred"])

-0.14117049439768015

In [9]:
model.save()

In [10]:
preds = model.predict(df_test)
preds

No rows removed. All molecules are valid.
Using cached descriptor values for RDKitGEOM
Using cached descriptor values for RDKitAUTOCORR
Using cached descriptor values for RDKitRDF
Using cached descriptor values for RDKitMORSE
Using cached descriptor values for RDKitWHIM
Using cached descriptor values for MolFeatUSRD
Using cached descriptor values for MolFeatElectroShape
Using cached descriptor values for RDKitGETAWAY
Using cached descriptor values for MolFeatPmapper


,SMILES,pred
0,C[C@H]1c2c(cc(F)c(-c3cccc4c(Cl)c[nH]c34)c2F)NC...,-0.476398
1,CNC(=O)C[C@H]1COc2cc(F)c(CC(C)C)cc2N1C(=O)c1cc...,-2.145782
2,CNC(=O)c1cccc(CC[C@]2(O)CCC3=Cc4c(cnn4-c4ccc(F...,-0.755328
3,CC(=O)[C@@]1(O)CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@...,-3.595974
4,Cn1cc(S(=O)(=O)N2CC[C@H]3Cc4c(cnn4-c4ccc(F)cc4...,-0.918757
5,C[C@]12C[C@H](O)[C@@]3(F)[C@@H](C[C@H](F)C4=CC...,-0.163066
6,COc1c(O)ccc2c1-c1ccc3c(c1/C(=C/c1sccc1C(O)C(F)...,0.162163
7,C=C1C[C@H](C)CC(C)(C)[C@@H]1Cc1cc(OC)c(Br)cc1O,-3.912323
8,CC1=CC(C)(C)Nc2ccc3c(c21)/C(=C/c1ccc(F)c(F)c1)...,-0.897260
9,CC1=CC(C)(C)Nc2ccc3c(c21)C(CCC(C)C)Oc1ccccc1-3,-2.240073


In [11]:
r2_score(df_test["y"], preds["pred"])

-0.14117049439768015

In [12]:
model = MultiConformerModel.load("./mcfm/model.pkl")
model.predictFromSMILES(df_test["smiles"])

No rows removed. All molecules are valid.
Generating conformers: 15/15
Rebuilding trainer for InstanceWrapperMLPNetworkRegressor(
  (instance_transformer): Sequential(
    (0): Linear(in_features=11, out_features=256, bias=True)
    (1): GELU(approximate='none')
    (2): Linear(in_features=256, out_features=128, bias=True)
    (3): GELU(approximate='none')
    (4): Linear(in_features=128, out_features=64, bias=True)
    (5): GELU(approximate='none')
  )
  (bag_estimator): Linear(in_features=64, out_features=1, bias=True)
) with parameters: 
"accelerator":        cpu
"activation":         gelu
"batch_size":         128
"early_stopping":     True
"hidden_layer_sizes": (256, 128, 64)
"instance_dropout":   0.0
"learning_rate":      0.001
"max_epochs":         1000
"num_workers":        0
"pool":               mean
"random_seed":        42
"verbose":            False
"weight_decay":       0.0
Rebuilding trainer for BagWrapperMLPNetworkRegressor(
  (instance_transformer): Sequential(
    (0)

,SMILES,pred
0,C[C@H]1c2c(cc(F)c(-c3cccc4c(Cl)c[nH]c34)c2F)NC...,-0.476398
1,CNC(=O)C[C@H]1COc2cc(F)c(CC(C)C)cc2N1C(=O)c1cc...,-2.145782
2,CNC(=O)c1cccc(CC[C@]2(O)CCC3=Cc4c(cnn4-c4ccc(F...,-0.755328
3,CC(=O)[C@@]1(O)CC[C@H]2[C@@H]3CCC4=CC(=O)CC[C@...,-3.595974
4,Cn1cc(S(=O)(=O)N2CC[C@H]3Cc4c(cnn4-c4ccc(F)cc4...,-0.918757
5,C[C@]12C[C@H](O)[C@@]3(F)[C@@H](C[C@H](F)C4=CC...,-0.163066
6,COc1c(O)ccc2c1-c1ccc3c(c1/C(=C/c1sccc1C(O)C(F)...,0.162163
7,C=C1C[C@H](C)CC(C)(C)[C@@H]1Cc1cc(OC)c(Br)cc1O,-3.912323
8,CC1=CC(C)(C)Nc2ccc3c(c21)/C(=C/c1ccc(F)c(F)c1)...,-0.897260
9,CC1=CC(C)(C)Nc2ccc3c(c21)C(CCC(C)C)Oc1ccccc1-3,-2.240073


In [13]:
# works on unknown SMILES as well, no caching in that case
model.predictFromSMILES(
    [
        "CN1[C@H]2CC[C@@H]1[C@@H](C(OC)=O)[C@@H](OC(C3=CC=CC=C3)=O)C2",
        "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
    ]
)

No rows removed. All molecules are valid.
Generating conformers: 2/2


,SMILES,pred
0,CN1[C@H]2CC[C@@H]1[C@@H](C(OC)=O)[C@@H](OC(C3=...,-3.468972
1,CC(C)CC1=CC=C(C=C1)C(C)C(=O)O,-3.105506


In [14]:
# second call uses cached descriptors for the same SMILES, so it is faster
model.predictFromSMILES(
    [
        "CN1[C@H]2CC[C@@H]1[C@@H](C(OC)=O)[C@@H](OC(C3=CC=CC=C3)=O)C2",
        "CC(C)CC1=CC=C(C=C1)C(C)C(=O)O"
    ]
)

No rows removed. All molecules are valid.
Using cached descriptor values for RDKitGEOM
Using cached descriptor values for RDKitAUTOCORR
Using cached descriptor values for RDKitRDF
Using cached descriptor values for RDKitMORSE
Using cached descriptor values for RDKitWHIM
Using cached descriptor values for MolFeatUSRD
Using cached descriptor values for MolFeatElectroShape
Using cached descriptor values for RDKitGETAWAY
Using cached descriptor values for MolFeatPmapper


,SMILES,pred
0,CN1[C@H]2CC[C@@H]1[C@@H](C(OC)=O)[C@@H](OC(C3=...,-3.468972
1,CC(C)CC1=CC=C(C=C1)C(C)C(=O)O,-3.105506
